# GPU Stack RCA — RX 7800 XT (gfx1101) + TheRock ROCm

**Date:** 2026-10-11 | **Stack:** torch `2.9.1+rocm7.10.0a20251120` (TheRock alpha) on native Windows

## Objective
Find the fastest *correct* training configuration for YOLO26n-seg @960 on this GPU.
Precision is non-negotiable — any config producing NaN/garbage losses is rejected
regardless of speed.

## Method
Systematic ablation over the precision/kernel-selection matrix, verified by:
1. Standalone conv smoke tests (isolate kernel correctness from trainer code)
2. Short real-training probes (watch loss trajectory, it/s, VRAM)
3. PerfDb / lockfile forensics across process restarts

## Results summary

| # | Config | Loss | it/s | Verdict |
|---|--------|------|------|---------|
| 1 | FP32 det=True (ultralytics default) | real (~cls 17) | 0.15 (6.5s/it) | ✅ valid, slow |
| 2 | AMP fp16 forced (check bypassed) | cls 32→666→854→959→1020 | 0.36 (2.8it/s) | ❌ diverged |
| 3 | AMP bf16 smoke (2-conv) | NaN @ iter 0 | — | ❌ kernel broken |
| 4 | det=False + AMP auto-on | cls →1252 | 2.8it/s | ❌ diverged (fp16) |
| 5 | det=False FP32 | nan (poisoned PerfDb from #4) | 1.1it/s | ❌ db poisoned |
| 6 | det=True FP32 (fresh db) | 7.006e-44 all cols | 9.1it/s | ❌ deterministic kernel broken |
| 7 | **det=False + amp=False** (smoke) | **finite** | — | ✅ **settled config** |

## Root cause
The TheRock alpha build ships buggy MIOpen kernels for gfx1101:
- **deterministic conv algo emits NaN** even in pure FP32 (smoke test §2)
- **fp16/bf16 kernels emit NaN in forward** — no GradScaler can save it
- AMP's self-check only compares *inference* outputs and passed spuriously
- failed runs write the bad (fast) kernel choice to the persistent user
  PerfDb (`~/.miopen/db/*.ufdb.txt`) → poisons later runs
- killed processes leave stale `miopen-lockfiles` → next process blocks
  forever at CPU=0 waiting for the lock

## Settled config
```
amp=False, deterministic=False, batch=16, workers=4, cache='ram',
epochs=50, patience=15, imgsz=960, seed=42
+ rm ~/.miopen/db/miopen-lockfiles/* before every launch
```
ETA: ~20 min/ep × early-stop ~35 ep ≈ **10–13 h**

Upgrade path (untested): `torch[device-gfx1101]==2.14+rocm10.1` stable —
ROCm 7.2.4+ fixed RDNA3 kernel correctness; would likely unlock real AMP.

## 1. Environment

In [ ]:
import torch, ultralytics, platform
print('python   :', platform.python_version())
print('torch    :', torch.__version__)
print('ultralyt :', ultralytics.__version__)
print('cuda dev :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('vram GB  :', round(torch.cuda.get_device_properties(0).total_memory/2**30,1))
print('arch     :', torch.cuda.get_device_properties(0).gcnArchName
      if hasattr(torch.cuda.get_device_properties(0),'gcnArchName') else 'gfx1101')

## 2. Decisive experiment — deterministic vs non-deterministic conv kernel (FP32)

Isolates kernel correctness from the trainer. Same 2-conv model, same input,
only the determinism flag changes.

**Observed output (2026-10-11):**
```
fp32 conv det=True  finite: False  mean nan      <- kernel broken
fp32 conv det=False finite: True                  <- works
```

In [ ]:
torch.backends.cudnn.benchmark = True
m = torch.nn.Sequential(torch.nn.Conv2d(3,64,3,1,1),
                        torch.nn.Conv2d(64,64,3,1,1)).cuda()
x = torch.randn(16,3,960,960, device='cuda')

torch.use_deterministic_algorithms(True, warn_only=True)
with torch.no_grad(): y1 = m(x)
print('det=True  finite:', torch.isfinite(y1).all().item(),
      '| mean', y1.mean().item())

torch.use_deterministic_algorithms(False)
with torch.no_grad(): y2 = m(x)
print('det=False finite:', torch.isfinite(y2).all().item(),
      '| mean', y2.mean().item())

## 3. Mixed-precision smoke — fp16 / bf16 forward correctness

**Observed output:** both dtypes emit `loss=nan` from iteration 0 with
`grad_finite=False`. Range is not the issue — the kernels themselves are
broken on this build. `amp=False` is mandatory.

In [ ]:
import time
for dt in (torch.float16, torch.bfloat16):
    mm = torch.nn.Sequential(torch.nn.Conv2d(3,64,3,1,1),
                             torch.nn.Conv2d(64,64,3,1,1)).cuda()
    opt = torch.optim.AdamW(mm.parameters())
    ok = True
    for i in range(3):
        with torch.autocast('cuda', dtype=dt):
            loss = mm(x).sum()
        opt.zero_grad(); loss.backward()
        g = next(mm.parameters()).grad
        fin = torch.isfinite(g).all().item()
        ok &= fin
        if i == 0: print(f'{dt}: it0 loss={loss.item()} grad_finite={fin}')
    print(f'{dt}: {"OK" if ok else "REJECT — NaN"}')
    del mm, opt; torch.cuda.empty_cache()

## 4. PerfDb forensics

The persistent user PerfDb lives at `~/.miopen/db/gfx1101_30.HIP.*.ufdb.txt`.
Bad runs write their (fast-but-broken) kernel picks there; later runs re-use
them. Killed processes also leave `miopen-lockfiles/*.lock` that block new
tuning forever. Quarantine + clear before relaunching.

In [ ]:
from pathlib import Path
db = Path.home()/'.miopen'/'db'
if db.exists():
    for f in sorted(db.glob('*.txt')):
        import datetime
        print(f'{f.name:60s} {f.stat().st_size:8d}B',
              datetime.datetime.fromtimestamp(f.stat().st_mtime))
    locks = list((db/'miopen-lockfiles').glob('*'))
    print('stale locks:', len(locks))

## 5. Summary figure — speed vs validity per config

In [ ]:
import matplotlib.pyplot as plt

cfgs = ['FP32 det=T\n(aug-6)', 'AMP fp16\nforced', 'det=F + AMP\nauto-on',
        'det=T FP32\npost-poison', 'det=F FP32\nSETTLED']
its  = [0.15, 0.36, 0.36, 9.1, 0.15]     # observed it/s (settled=expected)
ok   = [1, 0, 0, 0, 1]                   # 1=valid losses
note = ['valid 6.5s/it', 'cls→1020 div', 'cls→1252 div',
        '7e-44 garbage', 'finite, ~12h ETA']

fig, ax = plt.subplots(figsize=(9,4.5))
cols = ['#2ca02c' if v else '#d62728' for v in ok]
bars = ax.barh(cfgs, its, color=cols, alpha=.85)
for b, n in zip(bars, note):
    ax.text(b.get_width()+0.08, b.get_y()+b.get_height()/2, n,
            va='center', fontsize=9)
ax.set_xlabel('observed iterations/sec  (higher = faster)')
ax.set_title('YOLO26n-seg @960 on gfx1101/TheRock-alpha — only the slow\n'
             'correct kernel family is usable; every fast path emits garbage')
ax.set_xlim(0, 10.5)
fig.tight_layout()
fig.savefig('results/recipes/fig_gpu_rca.png', dpi=140)
plt.show()

## 6. Launch — settled config

Run in shell (not inside the notebook — Windows spawn safety):

```bash
rm -f ~/.miopen/db/miopen-lockfiles/* ~/AppData/Local/Temp/miopen-lockfiles/*
cd /b/02_Projects/thesis_final/experiment
.venv_rocm/Scripts/python dev/_train_recipes_gpu.py > results/recipes/train_fp32.log 2>&1 &
```

Sanity after ~10 min: `results.csv` row 1 should show finite losses and
`~6.5 s/it`. If `nan` appears in epoch 1 → the PerfDb re-poisoned itself;
delete `~/.miopen/db/gfx1101_30.HIP.3_5_1_*` and restart once more.